In [ ]:
from scipy.stats import lognorm, norm, truncnorm
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, Javascript, display_html, HTML 

### -> Generate distributions and samples.

In [10]:
## generate distributions and samples.
## generate distribution plots for inputs.
## Start function to generate scatter plots for Inputs vs OGIP.

def GenSamples(Dist_Type, V_P90, V_P10, V_Low_limit, V_mode, V_High_limit, V_Constant):
    V_trials: float = 10000

    if Dist_Type == 'Uniform':
        Samples_arr = np.random.uniform(low=V_Low_limit, high=V_High_limit, size=V_trials)

    if Dist_Type == 'Triangular':
        Samples_arr = np.random.triangular(left=V_Low_limit, mode=V_mode,right= V_High_limit, size=V_trials)

    if Dist_Type == 'Normal':
        x1, x2 = V_P90, V_P10 #x1, x2 = 10, 90
        p1ppf, p2ppf = norm.ppf(0.10), norm.ppf(0.90)

        Mean = ((x1 * p2ppf) - (x2 * p1ppf)) / (p2ppf - p1ppf)  #Mean
        SD = (x2 - x1) / (p2ppf - p1ppf)                         # Standard Dev

        a,b = (V_Low_limit-Mean)/SD, (V_High_limit-Mean)/SD
        Samples_arr = truncnorm.rvs(a=a,b=b, loc=Mean, scale=SD, size=V_trials)
        #NT_Samples=truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)
        
    if Dist_Type == 'Log-Normal':
        x1, x2 = np.log(V_P90), np.log(V_P10) #x1, x2 = 10, 90
        p1ppf, p2ppf = norm.ppf(0.10), norm.ppf(0.90)

        Mean = ((x1 * p2ppf) - (x2 * p1ppf)) / (p2ppf - p1ppf)   #Mean
        SD = (x2 - x1) / (p2ppf - p1ppf)                         # Standard Dev

        a,b = (V_Low_limit-Mean)/SD, (V_High_limit-Mean)/SD
        #Samples_arr = truncnorm.rvs(a=a,b=b, loc=Mean, scale=SD, size=V_trials)
        Samples_arr = lognorm(s=SD, scale=np.exp(Mean)).rvs(size=V_trials)
        
    n_Samples, n_Mean, n_SD = Samples_arr, np.mean(Samples_arr), np.std(Samples_arr)
    
    return n_Samples, n_Mean, n_SD


In [4]:
###########################################################
# Start function to generate distribution plots for inputs

def Plot_distributions(Samples_arr, V_Title,V_color, n_row,n_col,ax):
    ax[n_row,n_col].hist(Samples_arr, bins=60, density=True, align='mid', color=V_color, alpha=0.55, edgecolor = "gray")
    ax[n_row,n_col].set_title(V_Title, weight='bold', fontsize = 18, pad=19)
    ax[n_row,n_col].set_xlabel(' ', fontsize = 18)
    
    
    y_limit = ax[n_row,n_col].get_ylim()
    ax[n_row,n_col].set_ylim(y_limit)
    
    ### Plot black dot Mean ###
    n_Mean = round(np.mean(Samples_arr),3)
    if n_Mean > 1:
        n_Mean = round(n_Mean,0)
        
    n_txt = 'Mean ' + str(n_Mean)
    ax[n_row,n_col].annotate(n_txt, xy=(n_Mean,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                             horizontalalignment='center')
    
    ax[n_row,n_col].scatter(n_Mean, y_limit[1], clip_on=False,color='black') 
    
    ### Plot black dot P90 ###
    pn = round(np.percentile(Samples_arr, 10),3)
    if pn > 1:
        pn = round(pn,0)
        
    n_txt = '(P90) ' + str(pn)
    ax[n_row,n_col].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                            horizontalalignment='right')
    
    ax[n_row,n_col].scatter(pn, y_limit[1], clip_on=False,color='black') 
    
    ### Plot black dot P10 ###
    pn = round(np.percentile(Samples_arr, 90),3)
    if pn > 1:
        pn = round(pn,0)
        
    n_txt = '(P10) ' +str(pn)
    ax[n_row,n_col].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                            horizontalalignment='left')
    
    ax[n_row,n_col].scatter(pn, y_limit[1], clip_on=False,color='black')
    
    ax[n_row,n_col].get_yaxis().set_ticks([])
    #ax[n_row,n_col].get_yaxis().set_visible(False)
    ax[n_row,n_col].set_ylabel('PD', fontsize = 12)

In [15]:
# Start function to generate scatter plots for Inputs vs OGIP

def Plot_Scatter(X_sample,Y_sample, y_text,V_color, xx,yy, axSC):
    axSC[xx,yy].scatter(x=X_sample, y=Y_sample, color=V_color, alpha=0.1, s = 70)
    axSC[xx,yy].set_xlabel('OGIP, Bcf', size=12, weight='bold')
    axSC[xx,yy].set_ylabel(y_text, size=12, weight='bold')
    
    #sns.kdeplot(x=X_sample, y=Y_sample, levels=6, fill=True, alpha=0.2, cut=2, ax=axSC[xx,yy], label='---');

In [14]:
def Plot_OGIP(OGIP_Samples,r_chart, x_Label, ax2):
    if r_chart == 0:
        alp = 0.5
    else:
        alp = 0.8
    N, bins, patches = ax2[r_chart,0].hist(OGIP_Samples, bins=60, edgecolor='black', linewidth=1, color='red',alpha=alp)
    
    # facecolor for each bar
    pn90 = round(np.percentile(OGIP_Samples, 10),3)
    pn10 = round(np.percentile(OGIP_Samples, 90),3)
    for i in range(len(N)):
        if (bins[i] < pn90) or (bins[i]>=pn10):
           patches[i].set_facecolor('yellow')

    ax2[r_chart,0].set_xlim(left=0)
    ax2[r_chart,0].set_xlabel(x_Label, fontsize = 18,weight='bold');

    y_limit = ax2[r_chart,0].get_ylim()
    ax2[r_chart,0].set_ylim(y_limit)

    ### Plot black dot Mean ###
    n_Mean = round(np.mean(OGIP_Samples),3)
    if n_Mean > 1:
        n_Mean = round(n_Mean,0)
        
    n_txt = str(n_Mean)
    ax2[r_chart,0].annotate(n_txt, xy=(n_Mean,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                             horizontalalignment='center')

    ax2[r_chart,0].annotate('Mean', xy=(n_Mean,y_limit[1]), xycoords='data', xytext=(0,17), textcoords='offset points',
                             horizontalalignment='center')
        
    ax2[r_chart,0].scatter(n_Mean, y_limit[1], clip_on=False,color='black')


    ### Plot black dot P90 ###
    pn = round(np.percentile(OGIP_Samples, 10),3)
    if pn > 1:
        pn = round(pn,0)
        
    n_txt = str(pn)
    ax2[r_chart,0].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                            horizontalalignment='center')

    ax2[r_chart,0].annotate('P90', xy=(pn,y_limit[1]), xycoords='data', xytext=(0,17), textcoords='offset points',
                            horizontalalignment='center')
        
    ax2[r_chart,0].scatter(pn, y_limit[1], clip_on=False,color='black') 
    
    ### Plot black dot P10 ###
    pn = round(np.percentile(OGIP_Samples, 90),3)
    max_val = np.max(OGIP_Samples)

    if pn > 1:
        pn = round(pn,0)
        
    n_txt = str(pn)
    ax2[r_chart,0].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                            horizontalalignment='center')
    n_txt = 'P10'
    ax2[r_chart,0].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,17), textcoords='offset points',
                            horizontalalignment='center')
        
    x_val = pn
    y_val = y_limit[1]
    ax2[r_chart,0].scatter(x_val, y_val, clip_on=False,color='black')
    
    
    
    #########################################################
    # Cumulative Plot
    Nc, binsc, patchesc = ax2[r_chart,1].hist(OGIP_Samples,cumulative=-1, bins=60, edgecolor='black', linewidth=1, 
                                   color='red',alpha=alp)
    axx2 = ax2[r_chart,1]
    # facecolor for each bar
    pn90, pn10 = round(np.percentile(OGIP_Samples, 10),3), round(np.percentile(OGIP_Samples, 90),3)
    max_ogip = np.max(binsc)
    
    for i in range(len(Nc)):
        if (binsc[i] < pn90) or (binsc[i]>=pn10):
           patchesc[i].set_facecolor('yellow')
        
        if binsc[i]<n_Mean:
            y_limit_M=Nc[i]

        if binsc[i]<pn90:
            y_limit_P90=Nc[i]

        if binsc[i]<pn10:
            y_limit_P10=Nc[i]
    
    axx2.set_xlim(left=0)
    axx2.set_xlabel(x_Label, fontsize = 18,weight='bold');
    
    dx = max_ogip/60
    x_vals, y_vals = [n_Mean, pn90+dx, pn10+dx], [y_limit_M,y_limit_P90,y_limit_P10]
    
    axx2.scatter(x_vals, y_vals, clip_on=False,color='black') # add dots for Mean, P90 and P10
    
    # P10 Annotations on plot
    Add_annotations('P10',pn10, y_limit_P10, dx,axx2)
    Add_annotations('Mean',n_Mean, y_limit_M, dx,axx2)
    Add_annotations('P90',pn90, y_limit_P90, dx,axx2)

In [ ]:
def Add_annotations(n_txt,x_val,y_val,dx,axx2):
    xx_val,yy_val = x_val+(dx*0), y_val
    axx2.annotate(n_txt, xy=(xx_val,yy_val), xycoords='data', xytext=(0,5), textcoords='offset pixels',
                  horizontalalignment='left', fontsize=9)
    
    n_txt,xx_val, yy_val = str(round(x_val,0)), x_val+(dx*0), y_val
    axx2.annotate(n_txt, xy=(xx_val,yy_val), xycoords='data', xytext=(0,15), textcoords='offset pixels',
                  horizontalalignment='left', fontsize=14, weight='bold')

### -> Generate distribution plots for inputs.

In [9]:

# Input data cell
def Generate_Samples_and_Plot_Distribution(Input_var_dict, ax):

    color_L = ['gray','orange', 'yellow','blue', 'lightcoral', 'red', 'gray','green', 'green']

    Input_df = pd.DataFrame()
    n_row, n_col, i = 0, 0, 0

    for key in Input_var_dict:
        Dist_Type = Input_var_dict[key][0]
        var_label = Input_var_dict[key][1]
        val_Const = Input_var_dict[key][2]
        low, high = Input_var_dict[key][3], Input_var_dict[key][4]
        val_Mode = Input_var_dict[key][5]
        val_P90, val_P10 = Input_var_dict[key][6], Input_var_dict[key][7]

        n_Samples, n_Mean, n_SD = GenSamples(Dist_Type, val_P90, val_P10, low, val_Mode, high,val_Const)
        Input_df[key] = n_Samples

        Plot_distributions(n_Samples, var_label, color_L[i], n_row, n_col,ax)

        i = i + 1
        n_col = n_col + 1
        if n_col == 3:
            n_row = n_row + 1
            n_col = 0
    
    # Set decimals to DF
    Input_df['Rock_Vol'] = Input_df['Rock_Vol'].apply(lambda x: round(x, 1))
    Input_df.NTG = Input_df.NTG.apply(lambda x: round(x, 3))
    Input_df.Porosity = Input_df.Porosity.apply(lambda x: round(x, 3))
    Input_df.Sw = Input_df.Sw.apply(lambda x: round(x, 3))
    Input_df.Eg = Input_df.Eg.apply(lambda x: round(x, 1))
    #Input_df.columns.names = ['Stats']


    Input_Stats_df = Input_df.describe(percentiles=[0.01,0.1,.25,.75,0.9,0.99])

    Input_Stats_df = Input_Stats_df.reset_index(drop=True)
    Input_Stats_df.index = ['Samples', 'Mean', 'SD', 'Min','P99','P90','P75','P50','P25','P10','P1','Max'] 
    Input_Stats_df.columns.names = ['Stats']

    Input_Stats_df.Rock_Vol, Input_Stats_df.NTG = Input_Stats_df.Rock_Vol.round(1), Input_Stats_df.NTG.round(4)
    Input_Stats_df.Porosity, Input_Stats_df.Sw = Input_Stats_df.Porosity.round(4), Input_Stats_df.Sw.round(4)
    Input_Stats_df.Eg = Input_Stats_df.Eg.round(1)
    Input_Stats_df.RFg = Input_Stats_df.RFg.round(3)
    Input_Stats_df.Merma = Input_Stats_df.Merma.round(3)
    Input_Stats_df.CGR = Input_Stats_df.CGR.round(1)
    Input_Stats_df.RFo = Input_Stats_df.RFo.round(3)
    


    # Apply formatting to the first row 
    ### Input_Stats_df.iloc[0] = Input_Stats_df.iloc[0].apply(format_decimals)

    Input_Stats_df

    return Input_df, Input_Stats_df

# Function to format decimals
def format_decimals(value):
    return f"{value:.0f}"


### -> Start function to generate scatter plots for Inputs vs OGIP.

In [13]:
# OGIP cell
def generate_DF_and_Format_Table(Input_Stats_df,OGIP_arr,Gas_Prod_arr,Gas_sale_arr,OOIP_arr):    
    ## convert arrays to DF
    OGIP_df = pd.DataFrame ({'OGIP_Bcf':OGIP_arr,'Gas_Prod_Bcf':Gas_Prod_arr, 'Gas_sale':Gas_sale_arr, 'OOIP':OOIP_arr})


    # specify which percentiles to show on tables
    Stats_Table_L = ['Samples', 'Mean', 'SD', 'Min','P99','P90','P75','P50','P25','P10','P1','Max'] 
    Percentiles_L = [0.01,0.1,.25,.75,0.9,0.99]

    OGIP_Stats_df = OGIP_df.describe(percentiles=Percentiles_L)

    OGIP_Stats_df = OGIP_Stats_df.reset_index(drop=True)
    OGIP_Stats_df.index = Stats_Table_L
    OGIP_Stats_df.columns.names = ['Stats']
    
    # Apply formatting to the first row
    #OGIP_Stats_df.iloc[0] = OGIP_Stats_df.iloc[0].apply(format_decimals)
    #numeric_cols = df.select_dtypes(include=['number']).columns
    #df.loc[0, numeric_cols] = df.loc[0, numeric_cols].apply(format_decimals)

    df1_styler =  OGIP_Stats_df.style.set_table_attributes("style='display:inline'").set_caption('Solution').format(precision=1)


    df2_styler = Input_Stats_df.style.set_table_attributes("style='display:inline'").set_caption('Input').format(precision=4)

    display_html(df1_styler._repr_html_() + df2_styler._repr_html_(), raw=True)
    
    return OGIP_df, OGIP_Stats_df


